# Component 3 Walkthrough — CRMA Risk Inference & GeoJSON Output

C3 combines exceedance probabilities from the Zarr store with near-real-time
GPM satellite observations to drive the Compound Risk Model for the IGAD region
(CRMA). Each admin-1 unit receives a four-state risk label (Green/Yellow/Orange/Red)
with marginal probabilities that sum to 1.

**Key classes**: `CRMAModel`, `CRMAEvidence`, `run_risk_batch`

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
import json, tempfile
from datetime import date

WORK_DIR = Path(tempfile.mkdtemp(prefix='gik_c3_'))

NLAT, NLON, NMEMBERS, NSTEPS = 8, 8, 10, 16
LAT   = np.linspace(0.0, 4.0, NLAT, dtype=np.float32)
LON   = np.linspace(35.0, 39.0, NLON, dtype=np.float32)
WINDOWS_H      = [24, 72, 168]
RETURN_PERIODS = [5, 20]
TEST_DATE      = date(2024, 10, 15)

## 3.1  Build the CRMA Bayesian Network

In [ ]:
try:
    import pgmpy
    from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

    model = CRMAModel()
    model.build()
    print('CRMA model built.')
    print('Nodes:', list(model._model.nodes()))
    CRMA_AVAILABLE = True
except ImportError:
    print('pgmpy not installed — install with: pip install pgmpy')
    CRMA_AVAILABLE = False

## 3.2  Single-Point Inference

In [ ]:
if CRMA_AVAILABLE:
    evidence = CRMAEvidence(
        exceedance_prob_24h_5y=0.35,
        exceedance_prob_72h_5y=0.28,
        exceedance_prob_7d_5y=0.20,
        gpm_obs_24h=20.0,
        api_mm=65.0,
        spatial_coverage_fraction=0.55,
        consecutive_signal_days=3,
        sat_consecutive_days=2,
    )
    result = model.infer(evidence)
    print(f"Risk: {result['risk_label']} (state {result['risk_state']})")
    for label in ['p_green', 'p_yellow', 'p_orange', 'p_red']:
        print(f'  {label}: {result[label]:.4f}')
    total = result['p_green'] + result['p_yellow'] + result['p_orange'] + result['p_red']
    assert abs(total - 1.0) < 1e-4
    print(f'Sum = {total:.6f} — OK')

## 3.3  Prepare Synthetic Input Stores

Builds a minimal exceedance Zarr store and GPM NetCDF file so `run_risk_batch` has all required inputs.

In [ ]:
rng      = np.random.default_rng(0)
p_data   = rng.uniform(0.0, 0.5, (1, NLAT, NLON, len(WINDOWS_H), len(RETURN_PERIODS))).astype(np.float32)
conf_data = rng.integers(0, 3, (1, NLAT, NLON), dtype=np.int8)

exc_ds = xr.Dataset(
    {
        'exceedance_prob': xr.DataArray(
            p_data,
            dims=['date', 'latitude', 'longitude', 'window', 'return_period'],
            coords={
                'date':          [pd.Timestamp(TEST_DATE)],
                'latitude':      LAT,
                'longitude':     LON,
                'window':        np.array(WINDOWS_H, dtype=np.int16),
                'return_period': np.array(RETURN_PERIODS, dtype=np.int16),
            },
        ),
        'ensemble_confidence': xr.DataArray(
            conf_data,
            dims=['date', 'latitude', 'longitude'],
            coords={'date': [pd.Timestamp(TEST_DATE)], 'latitude': LAT, 'longitude': LON},
        ),
    }
)
exc_store = str(WORK_DIR / 'exceedance.zarr')
exc_ds.to_zarr(exc_store, mode='w', consolidated=False)
print('Exceedance store written.')

gpm_dir = WORK_DIR / 'gpm'
gpm_dir.mkdir()
gpm_ds = xr.Dataset({'precipitationCal': xr.DataArray(
    rng.exponential(5.0, (NLAT, NLON)).astype(np.float32),
    dims=['lat', 'lon'],
    coords={'lat': LAT, 'lon': LON},
)})
gpm_file = gpm_dir / f"3B-DAY.MS.MRG.3IMERG.{TEST_DATE.strftime('%Y%m%d')}.V07B.nc4"
gpm_ds.to_netcdf(gpm_file)
print(f'GPM file written: {gpm_file.name}')

## 3.4  Admin Boundaries (synthetic GeoPackage)

In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import box

    gdf = gpd.GeoDataFrame(
        {
            'admin1_pcode': ['KE001', 'KE002'],
            'admin1_name':  ['West', 'East'],
            'country_code': ['KE', 'KE'],
        },
        geometry=[box(35.2, 0.2, 36.8, 1.8), box(37.0, 0.2, 38.5, 1.8)],
        crs='EPSG:4326',
    )
    admin_path = WORK_DIR / 'admin1.gpkg'
    gdf.to_file(admin_path, driver='GPKG')
    print('Admin boundaries written.')
    ADMIN_AVAILABLE = True
except ImportError:
    print('geopandas not installed — skipping admin boundaries')
    ADMIN_AVAILABLE = False

## 3.5  run_risk_batch → GeoJSON Output

In [ ]:
if CRMA_AVAILABLE and ADMIN_AVAILABLE:
    try:
        from gik_icechain.risk.risk_engine import run_risk_batch

        output_dir = WORK_DIR / 'risk_output'
        written = run_risk_batch(
            exceedance_store_uri=exc_store,
            gpm_dir=gpm_dir,
            admin_boundaries_path=admin_path,
            crma_model=model,
            output_dir=output_dir,
            start=TEST_DATE,
            end=TEST_DATE,
            endpoint_url=None,
        )

        assert len(written) == 1
        fc = json.loads(written[0].read_text())
        assert fc['type'] == 'FeatureCollection'

        for f in fc['features']:
            p     = f['properties']
            total = p['p_green'] + p['p_yellow'] + p['p_orange'] + p['p_red']
            assert abs(total - 1.0) < 1e-4
            print(f"  {p.get('admin1_pcode', '?')}: {p['risk_label']}  |"
                  f" G={p['p_green']:.3f} Y={p['p_yellow']:.3f} O={p['p_orange']:.3f} R={p['p_red']:.3f}")

        print(f'\nGeoJSON output: {written[0]}')
    except ImportError as e:
        print(f'Skipping risk batch: {e}')
else:
    print('Skipping: pgmpy or geopandas not available')